In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, pickle, os
warnings.filterwarnings("ignore")

from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.interpolate import PchipInterpolator


In [ ]:
FILE_PATH   = r"C:\Solar_SUDHA_maam\merged_output.csv"

LATITUDE    = 13.04
LONGITUDE   = 80.17
RATED_KW    = 250
AREA_EFF_M2 = 181.591

TRAIN_END = "2025-01-01"

IRR_PARAMS = dict(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.7, colsample_bytree=0.8,
    reg_alpha=1, reg_lambda=3, min_child_weight=10,
    tree_method="hist", random_state=42,
)

# Final power model — trained once on full train set
POWER_PARAMS = dict(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.7, colsample_bytree=0.8,
    reg_alpha=2, reg_lambda=5, min_child_weight=10,
    tree_method="hist", random_state=42,
)

# CV params — fewer trees, stops early, much less RAM
POWER_PARAMS_CV = dict(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.6, colsample_bytree=0.7,
    reg_alpha=2, reg_lambda=5, min_child_weight=15,
    tree_method="hist", random_state=42,
    early_stopping_rounds=15,
)


In [ ]:
df = pd.read_csv(FILE_PATH)
df.columns = df.columns.str.strip().str.lower()
df["time"] = pd.to_datetime(df["time"], dayfirst=True)
df = df.sort_values("time").reset_index(drop=True)

print(f"Rows: {len(df):,}  |  {df['time'].min().date()} → {df['time'].max().date()}")
df.head()


In [ ]:
# Time features
df["hour"]      = df["time"].dt.hour + df["time"].dt.minute / 60
df["dayofyear"] = df["time"].dt.dayofyear
df["month"]     = df["time"].dt.month

for col, period in [("hour", 24), ("dayofyear", 365), ("month", 12)]:
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)

# Solar geometry — computed from lat/lon/time only, no sensor needed
lat_rad     = np.radians(LATITUDE)
declination = np.radians(23.45) * np.sin(np.radians((360 / 365) * (df["dayofyear"] - 81)))
hour_angle  = np.radians(15 * (df["hour"] - 12))
sin_elev    = (np.sin(lat_rad) * np.sin(declination)
             + np.cos(lat_rad) * np.cos(declination) * np.cos(hour_angle))

df["solar_elevation"] = np.degrees(np.arcsin(np.clip(sin_elev, -1, 1)))
df["clear_sky_irr"]   = np.clip(1000 * sin_elev, 0, 1100)   # W/m²
df["clear_sky_power"] = df["clear_sky_irr"] * AREA_EFF_M2 / 1000  # kW ceiling

# Cloud factor: how much cloud attenuates clear-sky irradiance (0=overcast, 1=clear)
# Only defined when sun is above horizon
df["cloud_factor"] = np.where(
    df["clear_sky_irr"] > 10,
    (df["irradiance"] / df["clear_sky_irr"]).clip(0, 1.2),
    np.nan
)

print("Cloud factor stats:")
print(df["cloud_factor"].describe().round(3))


In [ ]:
# Tag each row with its meteorological season (Chennai-specific)
# Used later to check if test set covers all seasons
def get_season(month):
    if month in [6, 7, 8, 9]:   return "monsoon"
    if month in [10, 11]:        return "post-monsoon"
    if month in [12, 1, 2]:      return "winter"
    return "summer"

df["season"] = df["month"].map(get_season)

# Quick check — cloud factor by season reveals why old model failed on monsoon test
season_stats = df.groupby("season")["cloud_factor"].agg(["mean","std","count"]).round(3)
print("Cloud factor by season (shows distribution shift between seasons):")
print(season_stats)


In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)
before = len(df)
df = df.dropna(subset=["power", "temp", "humidity", "wind_speed", "precipitation"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with NaN  |  {len(df):,} remain")
print(f"Power range: [{df['power'].min():.2f}, {df['power'].max():.2f}] kW")


In [ ]:
train = df[df["time"] <  TRAIN_END].copy()
test  = df[df["time"] >= TRAIN_END].copy()

print(f"Train: {len(train):>7,} rows  |  {train['time'].min().date()} → {train['time'].max().date()}")
print(f"Test : {len(test):>7,} rows  |  {test['time'].min().date()} → {test['time'].max().date()}")

# Make sure every season is represented in training
print("\nSeasons in TRAIN:", train["season"].value_counts().to_dict())
print("Seasons in TEST :", test["season"].value_counts().to_dict())


In [ ]:
WEATHER_TIME_FEATS = [
    "temp", "humidity", "wind_speed", "precipitation",
    "hour_sin", "hour_cos",
    "dayofyear_sin", "dayofyear_cos",
    "month_sin", "month_cos",
    "solar_elevation",
]

POWER_FEATS = WEATHER_TIME_FEATS + [
    "clear_sky_irr",
    "clear_sky_power",
    "irr_est",
    "theo_est",
    "cf_est",
]

print(f"Stage-1 features: {len(WEATHER_TIME_FEATS)}")
print(f"Stage-2 features: {len(POWER_FEATS)}")


In [ ]:
cf_train = train.dropna(subset=["cloud_factor"]).copy()
print(f"Fitting Stage-1 on {len(cf_train):,} daytime training rows...")

irr_model = XGBRegressor(**IRR_PARAMS)
irr_model.fit(cf_train[WEATHER_TIME_FEATS], cf_train["cloud_factor"])

def add_irr_estimates(ds, model):
    ds = ds.copy()
    ds["cf_est"]   = np.clip(model.predict(ds[WEATHER_TIME_FEATS]), 0, 1.2)
    ds["irr_est"]  = (ds["clear_sky_irr"] * ds["cf_est"]).clip(0, 1400)
    ds["theo_est"] = ds["irr_est"] * AREA_EFF_M2 / 1000
    return ds

train = add_irr_estimates(train, irr_model)
test  = add_irr_estimates(test,  irr_model)

print("\nStage-1 irradiance R² (daytime rows only):")
for name, ds in [("Train", train), ("Test ", test)]:
    mask   = ds["clear_sky_irr"] > 10
    actual = ds.loc[mask, "irradiance"]
    est    = ds.loc[mask, "irr_est"]
    r2  = r2_score(actual, est)
    mae = mean_absolute_error(actual, est)
    print(f"  {name}  R²={r2:.4f}   MAE={mae:.1f} W/m²")


In [ ]:
# CV uses POWER_PARAMS_CV (light) — for generalisation estimate only
# The final model below uses the full POWER_PARAMS
tscv = TimeSeriesSplit(n_splits=3)   # 3 folds instead of 5 — less RAM, still valid
cv_r2, cv_mae = [], []

X_tr_all = train[POWER_FEATS]
y_tr_all  = train["power"].fillna(0)

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_tr_all)):
    m = XGBRegressor(**POWER_PARAMS_CV)
    m.fit(
        X_tr_all.iloc[tr_idx], y_tr_all.iloc[tr_idx],
        eval_set=[(X_tr_all.iloc[val_idx], y_tr_all.iloc[val_idx])],
        verbose=False,
    )
    p   = np.clip(m.predict(X_tr_all.iloc[val_idx]), 0, None)
    r2  = r2_score(y_tr_all.iloc[val_idx], p)
    mae = mean_absolute_error(y_tr_all.iloc[val_idx], p)
    cv_r2.append(r2); cv_mae.append(mae)
    print(f"  Fold {fold+1}:  R²={r2:.4f}   MAE={mae:.3f} kW")

print(f"\n  Mean R²  = {np.mean(cv_r2):.4f} ± {np.std(cv_r2):.4f}")
print(f"  Mean MAE = {np.mean(cv_mae):.3f} ± {np.std(cv_mae):.3f} kW")


In [ ]:
power_model = XGBRegressor(**POWER_PARAMS)
power_model.fit(
    train[POWER_FEATS], train["power"].fillna(0),
    eval_set=[(test[POWER_FEATS], test["power"].fillna(0))],
    verbose=100,
)
print("Done.")


In [ ]:
def evaluate(y_true, y_pred, label=""):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1))) * 100
    print(f"{label}")
    print(f"  R²={r2:.4f}   MAE={mae:.3f} kW   RMSE={rmse:.3f} kW ({rmse/RATED_KW*100:.2f}% rated)   MAPE={mape:.1f}%")
    return dict(r2=r2, mae=mae, rmse=rmse, mape=mape)

train["predicted_power"] = np.clip(power_model.predict(train[POWER_FEATS]), 0, None)
test["predicted_power"]  = np.clip(power_model.predict(test[POWER_FEATS]),  0, None)

print("="*65)
evaluate(train["power"].fillna(0), train["predicted_power"], "TRAIN")
print()
evaluate(test["power"].fillna(0),  test["predicted_power"],  "TEST")
print()
# Daytime-only test (avoids nighttime zeros inflating R²)
td = test[test["clear_sky_irr"] > 10]
evaluate(td["power"].fillna(0), td["predicted_power"], "TEST — daytime only")


In [ ]:
test["month_name"] = test["time"].dt.strftime("%Y-%m")
monthly = {}
for m, grp in test.groupby("month_name"):
    y, p = grp["power"].fillna(0), grp["predicted_power"]
    monthly[m] = {"R²": round(r2_score(y, p), 3), "MAE": round(mean_absolute_error(y, p), 2)}

monthly_df = pd.DataFrame(monthly).T
print(monthly_df.to_string())


In [ ]:
fi = pd.Series(power_model.feature_importances_, index=POWER_FEATS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
fi.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Stage-2 power model — feature importance")
ax.set_xlabel("XGBoost gain")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()

print(fi.sort_values(ascending=False).round(4).to_string())


In [ ]:
# Pick one day from each season in the test set
season_samples = {}
for season in ["summer", "monsoon", "post-monsoon", "winter"]:
    rows = test[(test["season"] == season) & (test["clear_sky_irr"] > 100)]
    if len(rows):
        season_samples[season] = rows["time"].dt.date.iloc[0]

sample_dates = list(season_samples.values())[:3]

fig, axes = plt.subplots(len(sample_dates), 1, figsize=(13, 4 * len(sample_dates)), sharex=False)
if len(sample_dates) == 1: axes = [axes]

for ax, d in zip(axes, sample_dates):
    g = test[test["time"].dt.date == d]
    ax.plot(g["time"], g["power"],           lw=1.8, label="Actual")
    ax.plot(g["time"], g["predicted_power"], lw=1.5, ls="--", label="Predicted")
    ax.plot(g["time"], g["clear_sky_power"], lw=1,   ls=":",  color="orange", alpha=0.7, label="Clear-sky")
    ax.set_title(str(d)); ax.set_ylabel("Power (kW)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

plt.suptitle("Actual vs Predicted — test set sample days", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(test["power"].fillna(0), test["predicted_power"], alpha=0.15, s=3, rasterized=True)
lim = max(test["power"].max(), test["predicted_power"].max()) * 1.05
ax.plot([0, lim], [0, lim], "r--", lw=1.2, label="Perfect")
ax.set_xlabel("Actual (kW)"); ax.set_ylabel("Predicted (kW)")
ax.set_title("Test set: actual vs predicted"); ax.legend(); ax.grid(alpha=0.3)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
plt.tight_layout(); plt.show()


In [ ]:
test["residual"] = test["power"].fillna(0) - test["predicted_power"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Residuals vs time
axes[0].scatter(test["time"], test["residual"], alpha=0.1, s=2, rasterized=True)
axes[0].axhline(0, color="red", lw=0.8)
axes[0].set_xlabel("Time"); axes[0].set_ylabel("Residual (kW)")
axes[0].set_title("Residuals over time"); axes[0].grid(alpha=0.3)

# Residuals by hour of day — reveals systematic bias at specific times
hourly_resid = test.groupby(test["time"].dt.hour)["residual"].mean()
axes[1].bar(hourly_resid.index, hourly_resid.values, color="steelblue", alpha=0.8)
axes[1].axhline(0, color="red", lw=0.8)
axes[1].set_xlabel("Hour of day"); axes[1].set_ylabel("Mean residual (kW)")
axes[1].set_title("Mean residual by hour"); axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout(); plt.show()

print(f"Mean residual: {test['residual'].mean():.3f} kW  (should be close to 0)")
print(f"Std residual : {test['residual'].std():.3f} kW")


In [ ]:
def predict_power(timestamps, temp, humidity, wind_speed, precipitation):
    """Weather forecast → power (kW). No irradiance sensor needed."""
    inp = pd.DataFrame({
        "time": pd.to_datetime(timestamps),
        "temp": temp, "humidity": humidity,
        "wind_speed": wind_speed, "precipitation": precipitation,
    })
    inp["hour"]      = inp["time"].dt.hour + inp["time"].dt.minute / 60
    inp["dayofyear"] = inp["time"].dt.dayofyear
    inp["month"]     = inp["time"].dt.month
    for col, period in [("hour", 24), ("dayofyear", 365), ("month", 12)]:
        inp[f"{col}_sin"] = np.sin(2 * np.pi * inp[col] / period)
        inp[f"{col}_cos"] = np.cos(2 * np.pi * inp[col] / period)

    lat_r = np.radians(LATITUDE)
    decl  = np.radians(23.45) * np.sin(np.radians((360/365) * (inp["dayofyear"] - 81)))
    ha    = np.radians(15 * (inp["hour"] - 12))
    se    = np.sin(lat_r)*np.sin(decl) + np.cos(lat_r)*np.cos(decl)*np.cos(ha)
    inp["solar_elevation"] = np.degrees(np.arcsin(np.clip(se, -1, 1)))
    inp["clear_sky_irr"]   = np.clip(1000 * se, 0, 1100)
    inp["clear_sky_power"] = inp["clear_sky_irr"] * AREA_EFF_M2 / 1000

    inp["cf_est"]   = np.clip(irr_model.predict(inp[WEATHER_TIME_FEATS]), 0, 1.2)
    inp["irr_est"]  = (inp["clear_sky_irr"] * inp["cf_est"]).clip(0, 1400)
    inp["theo_est"] = inp["irr_est"] * AREA_EFF_M2 / 1000

    inp["predicted_power"] = np.clip(power_model.predict(inp[POWER_FEATS]), 0, None)
    inp.loc[inp["solar_elevation"] <= 0, "predicted_power"] = 0.0

    return inp[["time", "predicted_power", "irr_est", "cf_est", "clear_sky_power"]]


In [ ]:
data = [
    ("2026-02-10T23:00", 28.1, 80,  6.1, 0  ),
    ("2026-02-11T00:00", 27.8, 81,  4.4, 0  ),
    ("2026-02-11T01:00", 27.5, 82,  4.8, 0.1),
    ("2026-02-11T02:00", 27.5, 82,  4.9, 0  ),
    ("2026-02-11T03:00", 26.9, 85,  4.0, 0  ),
    ("2026-02-11T04:00", 26.5, 88,  5.4, 0  ),
    ("2026-02-11T05:00", 26.5, 89,  5.6, 0.1),
    ("2026-02-11T06:00", 27.5, 83,  4.8, 0  ),
    ("2026-02-11T07:00", 28.5, 77,  6.2, 0.1),
    ("2026-02-11T08:00", 29.5, 73, 10.2, 0.1),
    ("2026-02-11T09:00", 29.5, 71, 11.0, 1.6),
    ("2026-02-11T10:00", 29.3, 73, 10.2, 0.4),
    ("2026-02-11T11:00", 31.2, 64, 13.3, 0  ),
    ("2026-02-11T12:00", 31.5, 64, 13.7, 0  ),
    ("2026-02-11T13:00", 31.8, 63, 14.4, 0  ),
    ("2026-02-11T14:00", 31.5, 64, 15.2, 0  ),
    ("2026-02-11T15:00", 31.1, 65, 14.8, 0  ),
    ("2026-02-11T16:00", 30.2, 70, 14.9, 0  ),
    ("2026-02-11T17:00", 29.2, 71, 13.4, 0  ),
    ("2026-02-11T18:00", 28.1, 78, 11.3, 0  ),
    ("2026-02-11T19:00", 28.0, 81,  8.9, 0  ),
    ("2026-02-11T20:00", 27.8, 83,  7.6, 0  ),
    ("2026-02-11T21:00", 27.6, 84,  5.7, 0  ),
    ("2026-02-11T22:00", 27.1, 87,  4.4, 0  ),
    ("2026-02-11T23:00", 27.0, 87,  5.4, 0  ),
    ("2026-02-12T00:00", 27.0, 86,  5.5, 0  ),
]

hourly = pd.DataFrame(data, columns=["time","temp","humidity","wind_speed","precipitation"])
hourly["time"] = pd.to_datetime(hourly["time"])

# Expand hourly → 5-min via PCHIP
t_5min = pd.date_range("2026-06-11 05:00", "2026-06-11 19:00", freq="5min")
t_h = hourly["time"].astype("int64") // 10**9
t_q = t_5min.astype("int64") // 10**9
five_min = pd.DataFrame({"time": t_5min})

for col in ["temp", "humidity", "wind_speed"]:
    fn = PchipInterpolator(t_h, hourly[col].values, extrapolate=False)
    vals = fn(t_q)
    vals = np.where(np.isnan(vals), np.interp(t_q, t_h, hourly[col].values), vals)
    five_min[col] = vals.round(3)

five_min["humidity"]   = five_min["humidity"].clip(0, 100)
five_min["wind_speed"] = five_min["wind_speed"].clip(0)

prec_5min = np.zeros(len(t_5min))
for _, row in hourly.iterrows():
    mask = (five_min["time"].dt.floor("h") == row["time"].floor("h")).values
    if mask.sum() > 0 and row["precipitation"] > 0:
        prec_5min[mask] = row["precipitation"] / mask.sum()
five_min["precipitation"] = np.round(prec_5min, 4)

result = predict_power(
    five_min["time"], five_min["temp"].values,
    five_min["humidity"].values, five_min["wind_speed"].values,
    five_min["precipitation"].values,
)

daytime   = result[result["clear_sky_power"] > 0.5]
peak_row  = daytime.loc[daytime["predicted_power"].idxmax()]
total_kwh = result["predicted_power"].sum() * 5 / 60

print(f"Peak power        : {peak_row['predicted_power']:.2f} kW at {peak_row['time'].strftime('%H:%M')}")
print(f"Clear-sky ceiling : {daytime['clear_sky_power'].max():.2f} kW")
print(f"Total energy      : {total_kwh:.2f} kWh")
print(f"Mean cloud factor : {daytime['cf_est'].mean():.3f}")
print(f"Mean irr estimate : {daytime['irr_est'].mean():.0f} W/m²")

# Plot
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

ax = axes[0]
ax.fill_between(result["time"], 0, result["predicted_power"], alpha=0.25, color="steelblue")
ax.plot(result["time"], result["predicted_power"], lw=2, color="steelblue", label="Predicted power")
ax.plot(result["time"], result["clear_sky_power"], lw=1.2, ls=":", color="orange", label="Clear-sky ceiling")
ax.axhline(RATED_KW, color="red", lw=0.8, ls="--", alpha=0.5, label=f"Rated {RATED_KW} kW")
ax.set_ylabel("Power (kW)"); ax.set_title("2026-06-11 forecast — VIT Chennai 250 kW")
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_ylim(0)

ax2 = axes[1]
c1, c2 = "darkorange", "seagreen"
ln1 = ax2.plot(result["time"], result["irr_est"], color=c1, lw=1.8, label="Est. irradiance (W/m²)")
ax2.set_ylabel("Irradiance (W/m²)", color=c1); ax2.tick_params(axis="y", labelcolor=c1)
ax2r = ax2.twinx()
ln2 = ax2r.plot(result["time"], result["cf_est"], color=c2, lw=1.5, ls="--", label="Cloud factor")
ax2r.set_ylabel("Cloud factor", color=c2); ax2r.set_ylim(0, 1.25); ax2r.tick_params(axis="y", labelcolor=c2)
ax2.legend(ln1+ln2, ["Est. irradiance (W/m²)", "Cloud factor"], fontsize=9, loc="upper left")
ax2.grid(alpha=0.3); ax2.set_ylim(0)

ax3 = axes[2]
ax3.plot(five_min["time"], five_min["temp"],       lw=1.5, label="Temp (°C)")
ax3.plot(five_min["time"], five_min["humidity"],   lw=1.5, ls="--", label="Humidity (%)")
ax3.plot(five_min["time"], five_min["wind_speed"], lw=1.5, ls="-.", label="Wind (km/h)")
ax3r = ax3.twinx()
ax3r.bar(five_min["time"], five_min["precipitation"], width=pd.Timedelta("4min"), color="dodgerblue", alpha=0.4)
ax3r.set_ylabel("Precip (mm)", color="dodgerblue"); ax3r.tick_params(axis="y", labelcolor="dodgerblue")
ax3.set_ylabel("Weather"); ax3.set_xlabel("Time (IST)"); ax3.legend(fontsize=9); ax3.grid(alpha=0.3)

plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha="right")
fig.autofmt_xdate(rotation=30)
plt.tight_layout(); plt.show()


In [ ]:
save_dir = r"C:\Solar_SUDHA_maam\models"
os.makedirs(save_dir, exist_ok=True)

irr_path   = os.path.join(save_dir, "irr_model_v3.pkl")
power_path = os.path.join(save_dir, "power_model_v3.pkl")

with open(irr_path,   "wb") as f: pickle.dump(irr_model,   f)
with open(power_path, "wb") as f: pickle.dump(power_model, f)

print(f"Saved → {irr_path}")
print(f"Saved → {power_path}")
